In [ ]:
import os
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import scipy.stats as stats
import pandas as pd
from causalgraphicalmodels import CausalGraphicalModel

__Probabilistic Programming. Wasowski. Pardo. IT University of Copenhagen__

This file contains the list of exercises for the week, as well as any related code.

# Exercises

The exercises for this week are: all exercises from Chapter 9, McElreath, with exceptions and remarks noted below.

* __9E1, 9E2, 9E3, 9E4, 9E5, 9E6__ discussion questions
* __9E7__ skip the exercise. Instead modify the code in the lecture to display a __rank plot__ instead of a trace plot, and interpret it. Search here for "rank plot" to see how to switch to a rank plot: https://python.arviz.org/en/v0.17.0/api/generated/arviz.plot_trace.html . The rank plots made with arviz look a bit different than in the book (they are not overlayed), but the semantics seem to be the same. In a nutshell, rank plots show a histogram of the ranked posterior samples. If all chains target teh same posterior ranks should appear uniform.

* __9M1__ The exercise uses the terrain ruggedness model for the chapter.  I report it here after https://github.com/pymc-devs/resources/tree/main/Rethinking_2 for your convenience.

In [ ]:
d = pd.read_csv("Data/rugged.csv", delimiter=";")

d["log_gdp"] = np.log(d["rgdppc_2000"])
dd = d.dropna(subset=["log_gdp"]).copy()

dd["log_gdp_std"] = dd["log_gdp"] / dd["log_gdp"].mean()
dd["rugged_std"] = dd["rugged"] / dd["rugged"].max()

In [ ]:
cid = pd.Categorical(dd["cont_africa"])

with pm.Model() as m_8_3:
    a = pm.Normal("a", 1, 0.2, shape=cid.categories.size)
    b = pm.Normal("b", 0, 0.3, shape=cid.categories.size)

    mu = a[cid.codes] + b[cid.codes] * (dd["rugged_std"] - 0.215)
    sigma = pm.Exponential("sigma", 1)

    log_gdp_std = pm.Normal("log_gdp_std", mu, sigma, observed = dd["log_gdp_std"])

    m_8_3_trace = pm.sample()

az.summary(m_8_3_trace, kind="all", round_to=2)

* __9M2__: use the code above
* __9M3__: we have done this a bit in the lecture. Recall that for HMC (and consequently NUTS) warm up period is defined in the `tune` parameter for the `sample` method.
* __9H1__: The model code in PyMC, courtesy of https://github.com/pymc-devs/resources/tree/main/Rethinking_2

In [ ]:
with pm.Model() as H1:
    a = pm.Normal("a", 0, 0.1)
    b = pm.Cauchy("b", 0, 1)
    y = 1
    prior_sample = pm.sample_prior_predictive(samples=1000)
    H1_post_sample = pm.sample(draws=1000, chains=1)

* __9H2__: The models are to be found in multiple-lecture.ipynb; search for `m_5_1` and `m_5_3` (just compare these two models, skip model number 2).  We already have done the exercise in the lecture. Can you see why?

* __9H3__: The model is in the confounds-lecture.ipynb; search for `m_6_1`. To add a constraint to a normal distribution in PyMC add a model entry: `BoundedNormal = pm.Bound(pm.Normal, lower=0.0)` and then you can use `BoundedNormal` like normal (the constructor takes the same parameters). 

* __9H4__: Recall that `arviz.compare` is the function that can compute information criteria for a sample

* __9H5__: We have already done something similar in the lecture, but try other function population functions and observe whether you get the expected estimation of the posterior.

* Exercises __9H6__ and __9H7__ are more classic computer science exercises (implementing algorithms). They all require translating a piece of code in the chapter to Python and then making some modifications to it.  I believe that translating this manually is valuable because you get involved with the algorithm better but if you want to save time, this notebook should have the code: https://github.com/pymc-devs/resources/blob/main/Rethinking_2/Chp_09.ipynb. 